In [1]:
# Load generated data from .npz
import numpy as np
import os, glob, pickle
import numpy as np
from IPython.display import Image
import pypianoroll
import matplotlib.pyplot as plt
%matplotlib inline
from pathlib import Path

from pathlib import Path
OUT_DIR = Path("./output_midi")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("[info] Output dir:", OUT_DIR.resolve())

npz_path = "/data/generated_data/generated_drum_patterns.npz"
data = np.load(npz_path, allow_pickle=True)

# --- Inspect what's inside the NPZ ---
print("[info] NPZ path:", npz_path)
print("[info] Keys in file:", data.files)  # list of array names, e.g. ['drum_bin'] or ['arr_0']

for k in data.files:
    arr = data[k]
    print(f"\n[info] Key: {k}")
    print("  type:", type(arr))
    if hasattr(arr, "shape"):
        print("  shape:", arr.shape)
    if hasattr(arr, "dtype"):
        print("  dtype:", arr.dtype)

    # If it's an object array, peek inside safely
    if isinstance(arr, np.ndarray) and arr.dtype == object:
        try:
            sample = arr.flat[0]
            print("  object sample type:", type(sample))
            if isinstance(sample, dict):
                print("  object sample (dict) keys:", list(sample.keys())[:10])
        except Exception as e:
            print("  [warn] could not inspect object array sample:", e)

# OPTIONAL: pick the first array and show a tiny slice (to avoid huge prints)
if len(data.files) > 0:
    fk = data.files[0]
    A = data[fk]
    print(f"\n[peek] first key '{fk}':")
    try:
        # For typical drum predictions shaped like (B,46,16)
        if isinstance(A, np.ndarray) and A.ndim >= 2:
            # show a small corner so logs stay readable
            sl = tuple(slice(0, min(3, s)) for s in A.shape)  # up to 3 in each dim
            print("  small slice A", sl, "=\n", A[sl])
            # basic stats (safe for binary data)
            with np.errstate(all='ignore'):
                print("  min/max:", np.min(A), np.max(A))
                # Show unique values if small
                uniq = np.unique(A[:min(1000, A.size)])
                print("  unique (sampled):", uniq[:20], " ...")
        else:
            print("  value:", A)
    except Exception as e:
        print("  [warn] slice/summary failed:", e)

all_out = data["all_out"]  # generated drum predictions
all_tar = data["all_tar"]  # ground truth drums
all_idx = data["all_idx"]  # bar index strings
print(f"[info] Loaded {len(all_out)} generated bars from {npz_path}")
print(data)


[info] Output dir: /workspace/output_midi
[info] NPZ path: /data/generated_data/generated_drum_patterns.npz
[info] Keys in file: ['all_out', 'all_tar', 'all_idx']

[info] Key: all_out
  type: <class 'numpy.ndarray'>
  shape: (5120, 46, 16)
  dtype: float32

[info] Key: all_tar
  type: <class 'numpy.ndarray'>
  shape: (5120, 46, 16)
  dtype: float32

[info] Key: all_idx
  type: <class 'numpy.ndarray'>
  shape: (5120,)
  dtype: <U9

[peek] first key 'all_out':
  small slice A (slice(0, 3, None), slice(0, 3, None), slice(0, 3, None)) =
 [[[0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]]

 [[0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]]

 [[0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]]]
  min/max: 0.0 1.0
  unique (sampled): [0. 1.]  ...


[info] Loaded 5120 generated bars from /data/generated_data/generated_drum_patterns.npz
NpzFile '/data/generated_data/generated_drum_patterns.npz' with keys: all_out, all_tar, all_idx


# import library

# Ensure File DIR function

In [2]:
def ensure_dir(file_path):
    ed_directory = os.path.dirname(file_path)
    if not os.path.exists(ed_directory):
        os.makedirs(ed_directory)

# Read all song/bar index code

In [3]:
with open('/data/pre_processed_data/abs_bar_idx_str_list.pkl', 'rb') as pkl_file:      
    abs_bar_idx_str_list = pickle.load(pkl_file)
    
print ('[info] List of [song/bar] data is loaded.')
print ('[info] Total bars: {}'.format(len(abs_bar_idx_str_list)))
print ('[info] First 5 bar code: {}'.format(abs_bar_idx_str_list[:5]))
print ('[info] Last  5 bar code: {}'.format(abs_bar_idx_str_list[-5:]))


# Define function to get complete single song index (start, end)
song_index_in_list = np.unique([x.split('_')[0] for x in abs_bar_idx_str_list]).tolist()

def get_test_song_abs_idx(pick_song_index):

    song_index_all_bars = [x for x in abs_bar_idx_str_list if x[0:5]==song_index_in_list[pick_song_index]]
    bar_idx_start = abs_bar_idx_str_list.index(song_index_all_bars[0])
    bar_idx_end = abs_bar_idx_str_list.index(song_index_all_bars[-1])
    
    return ([bar_idx_start, bar_idx_end+1])


# for get_song_idx in range(0, 3):
for get_song_idx in range(len(song_index_in_list)):
    print('[info] Song idx: {:2d},   Start:{:4d},   End: {}'.format(get_song_idx,
                                                                    get_test_song_abs_idx(get_song_idx)[0],
                                                                    get_test_song_abs_idx(get_song_idx)[1]))

[info] List of [song/bar] data is loaded.
[info] Total bars: 5120
[info] First 5 bar code: ['00000_000', '00000_001', '00000_002', '00000_003', '00000_004']
[info] Last  5 bar code: ['00019_251', '00019_252', '00019_253', '00019_254', '00019_255']
[info] Song idx:  0,   Start:   0,   End: 256
[info] Song idx:  1,   Start: 256,   End: 512
[info] Song idx:  2,   Start: 512,   End: 768
[info] Song idx:  3,   Start: 768,   End: 1024
[info] Song idx:  4,   Start:1024,   End: 1280
[info] Song idx:  5,   Start:1280,   End: 1536
[info] Song idx:  6,   Start:1536,   End: 1792
[info] Song idx:  7,   Start:1792,   End: 2048
[info] Song idx:  8,   Start:2048,   End: 2304
[info] Song idx:  9,   Start:2304,   End: 2560
[info] Song idx: 10,   Start:2560,   End: 2816
[info] Song idx: 11,   Start:2816,   End: 3072
[info] Song idx: 12,   Start:3072,   End: 3328
[info] Song idx: 13,   Start:3328,   End: 3584
[info] Song idx: 14,   Start:3584,   End: 3840
[info] Song idx: 15,   Start:3840,   End: 4096
[in

# Reload all test result

In [ ]:
# Load generated drums directly from the NPZ you already mentioned in Cell 1
import numpy as np, os

def to_Bx46x16(arr: np.ndarray) -> np.ndarray:
    arr = np.asarray(arr)
    if arr.ndim == 3:
        # Expect (B,46,16) or (B,16,46)
        if arr.shape[1:] == (46, 16):
            return arr
        if arr.shape[1:] == (16, 46):
            return np.transpose(arr, (0, 2, 1))
        raise ValueError(f"3D pred has unsupported shape {arr.shape}")
    if arr.ndim == 4:
        # Something like (S,B,46,16) or (S,B,16,46) or (S,46,16,B) etc.
        axes = list(arr.shape)
        # find axes for 46 and 16
        try:
            ax46 = axes.index(46)
            ax16 = axes.index(16)
        except ValueError:
            raise ValueError(f"Could not find 46/16 axes in {arr.shape}")
        # remaining axes (song, bar) will be flattened to B
        other_axes = [i for i in range(4) if i not in (ax46, ax16)]
        # move to (other,46,16)
        arr = np.moveaxis(arr, (ax46, ax16), ( -2, -1))
        # now arr shape (*S,B*,46,16) -> flatten leading dims
        B = int(np.prod(arr.shape[:-2]))
        return arr.reshape(B, 46, 16)
    raise ValueError(f"Pred array has unsupported ndim={arr.ndim}, shape={arr.shape}")

with np.load(npz_path, allow_pickle=True) as d:
    # try common keys; else take first array
    for k in ("drum_bin", "drum_binary", "pred", "y_hat", "drum", "generated", "arr"):
        if k in d:
            GEN = d[k]; break
    else:
        GEN = list(d.values())[0]

drum_bin_all = to_Bx46x16(GEN).astype(np.uint8)  # shape (total_bars,46,16)
print(f"[info] predictions loaded from {npz_path}: bars={drum_bin_all.shape[0]} shape={drum_bin_all.shape}")

# The downstream notebook expects a list of "versions":
model_result_binary_list = [drum_bin_all]  # single version
add_note_ver_list = ["v00"]                # version tag
print(f"[info] reloaded 1 prediction set (v00).")


[info] predictions loaded from /data/generated_data/generated_drum_patterns.npz: bars=5120 shape=(5120, 46, 16)
[info] reloaded 1 prediction set (v00).


# Reload all original MIDI object

In [5]:
# class midi_track(object):
#     def __init__(self):
#         self.file_name = ""
#         self.pmidi_data = []
#         self.pmidi_all_tracks_data = []
#         self.pmidi_no_drum_data = []
#         self.pmidi_drum_only_data = []
#         self.tempo = 0        
#         self.downbeats_list_fixed = []
#         self.bar_range_list_fixed = []
#         self.drum_bar_list = []
#         self.drum_bar_list_bin = []
#         self.drum_bar_note_num = []
# #print ('MIDI track object is defined.')

# obj_file_name = '/data/pre_processed_data/proc_midi_object.pkl'
# with open(obj_file_name, 'rb') as pkl_file:
#     midi_obj_list = pickle.load(pkl_file)
    
# print('[info] All MIDI objects: {}'.format(len(midi_obj_list)))

# define original MIDI drum rebuild function (96, 128)

In [6]:
# keep 99 % of all instrument count (total 46 insts)
selected_inst_list_46 = [27, 28, 33, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, \
                         51, 53, 54, 55, 56, 57, 59, 60, 61, 62, 63, 64, 65, 67, 68, 69, 70, 73, \
                         74, 75, 76, 77, 80, 81, 82, 83, 85, 87]
print ('[info] # of keeped Insts: {}'.format(len(selected_inst_list_46)))

def get_odrum_shape(drum_ary_in):    
    odrum_data = np.zeros([96, 128])
    for x in range(0, drum_ary_in.shape[0]):
        for y in range(0, drum_ary_in.shape[1]):            
            pix_value = drum_ary_in[x,y]
            if pix_value>0.5:
                odrum_data[y*6, selected_inst_list_46[x]] = 100
            
    return (odrum_data)

print ('[info] get_odrum_shape is defined.')

[info] # of keeped Insts: 46
[info] get_odrum_shape is defined.


# load original midi data

In [7]:
# all_tracks_mid_flist = np.sort(glob.glob('./input_midi/my_input/*.mid', recursive=True)).tolist()
# all_tracks_mid_flist = np.sort([x.replace(' ','') for x in all_tracks_mid_flist if "all_tracks.mid" in x]).tolist()
# print ('[info] Total files: {}'.format(len(all_tracks_mid_flist)))
# for x in all_tracks_mid_flist[:]: print ('  ' + x)

# Build a map from song_id (first 5 chars) -> _all_tracks.mid path
# import os, glob, numpy as np

# ALL_TRACKS_DIR = './pre_processed_data/proc_all_tracks_mid'  # <-- this is where Step_1 wrote files
# all_tracks_mid_flist = np.sort(glob.glob(os.path.join(ALL_TRACKS_DIR, '*all_tracks.mid'))).tolist()
# print('[info] all_tracks files:', len(all_tracks_mid_flist))

# def extract_song_id_from_all_tracks_path(p):
#     base = os.path.basename(p).replace(' ', '')
#     if base.endswith('_all_tracks.mid'):
#         base = base[:-len('_all_tracks.mid')]
#     # Wei’s notebook derives song ids as first 5 chars from abs_bar_idx_str_list
#     return base[:5]

# songid_to_all_tracks = {}
# for p in all_tracks_mid_flist:
#     sid = extract_song_id_from_all_tracks_path(p)
#     songid_to_all_tracks[sid] = p

# print('[info] unique song ids in all_tracks:', len(songid_to_all_tracks))
# # Optionally, show a few:
# for i, (k, v) in enumerate(songid_to_all_tracks.items()):
#     if i >= 5: break
#     print('  ', k, '->', os.path.basename(v))

# Build a map: song_id (first 5 chars) -> Step-1 all_tracks MIDI path
import os, glob, numpy as np

ALL_TRACKS_DIR = './pre_processed_data/proc_all_tracks_mid'  # <-- Step-1 wrote here
all_tracks_mid_flist = np.sort(glob.glob(os.path.join(ALL_TRACKS_DIR, '*_all_tracks.mid'))).tolist()
print('[info] all_tracks files:', len(all_tracks_mid_flist))

def extract_song_id_from_all_tracks_path(p):
    base = os.path.basename(p).replace(' ', '')
    if base.endswith('_all_tracks.mid'):
        base = base[:-len('_all_tracks.mid')]
    return base[:5]  # Wei’s song_id convention

songid_to_all_tracks = {extract_song_id_from_all_tracks_path(p): p for p in all_tracks_mid_flist}
print('[info] unique song ids in all_tracks:', len(songid_to_all_tracks))
# peek a few
for i, (k, v) in enumerate(songid_to_all_tracks.items()):
    if i >= 5: break
    print('  ', k, '->', os.path.basename(v))

[info] all_tracks files: 0
[info] unique song ids in all_tracks: 0


# Loop all songs and save corresponding MIDI files

In [8]:
for x_idx, pick_song_index in enumerate(song_index_in_list):

    print ('[info] Start processing song: {} ...'.format(x_idx+1))

    # get complete single song index data
    abs_idx_start, abs_idx_end = get_test_song_abs_idx(x_idx)
    
    abs_song_idx = pick_song_index

    print ('[info] Song index: {}'.format(abs_song_idx))
    print ('[info] Song bars: {}'.format(abs_idx_end - abs_idx_start))
    print ('[info] start\end bar index:  {}\{}'.format(abs_idx_start, abs_idx_end))
    #print ('[info] Abs end index: {}'.format(abs_idx_end))
    #print('')

    # plot complete single song drum arrangement
    bar_idx_start = abs_idx_start
    bar_idx_end = abs_idx_end

    model_darr_odrm_ary_list = []
    
    for pch_ver in range(0, len(model_result_binary_list)):
    
        model_darr_list = []

        for bar_idx in range(bar_idx_start, bar_idx_end):
        
            plot_model_out_darr = model_result_binary_list[pch_ver][bar_idx,:,:]
        
            model_darr_list.append(plot_model_out_darr)
        

        # convert drum data into original shape (96, 128)
        model_darr_odrm_list = [get_odrum_shape(x) for x in model_darr_list]
        model_darr_odrm_ary = np.concatenate(model_darr_odrm_list, axis=0)
        #print(model_darr_odrm_ary.shape)

        model_darr_odrm_ary_list.append(model_darr_odrm_ary)
    
    print(f"[dbg] versions={len(model_darr_odrm_ary_list)}",
      f"first_version_shape={None if not model_darr_odrm_ary_list else model_darr_odrm_ary_list[0].shape}")
    
    #Get original NPZ file name
    # original_midi_file_path = all_tracks_mid_flist[x_idx]
    song_id = pick_song_index  # 5-char id used throughout
    original_midi_file_path = songid_to_all_tracks.get(song_id)
    if original_midi_file_path is None:
        # --- WRITE DRUM-ONLY even if we can't find all_tracks ---
        # Use the first version by default; change index if you prefer another version
        if len(model_darr_odrm_ary_list) == 0:
            print(f"[warn] no generated drum pianoroll for song_id={song_id}; skipping.")
        else:
            import pypianoroll as ppr
            pr = model_darr_odrm_ary_list[0]  # shape (T,128)
            drum_track = ppr.Track(pianoroll=pr, program=0, is_drum=True, name="Drums_Generated")
            drum_only = ppr.Multitrack(tracks=[drum_track], beat_resolution=24)

            out_path = Path("./output_midi") / f"{song_id}__drum_only.mid"
            out_path.parent.mkdir(parents=True, exist_ok=True)  # safety
            drum_only.write(str(out_path))
            print(f"[info] saved {out_path.name} (drum-only)")
        # And move on to the next song
        continue
    # pypiano_obj = pypianoroll.parse(original_midi_file_path, beat_resolution=24, name='original_track')
    pypiano_obj = pypianoroll.parse(original_midi_file_path)
    #ptymidi_obj = pypiano_obj.to_pretty_midi()
    mtrack_data = pypiano_obj
    
    for pch_idx in range(0, len(model_result_binary_list)):
        
        # write drum notes in multitrack object
        mtrack_data.append_track(track=None, 
                                 pianoroll=model_darr_odrm_ary_list[pch_idx], 
                                 program=pch_idx+1, 
                                 is_drum=True,
                                 name='Drums_{}'.format(add_note_ver_list[pch_idx]))

    # transfer data into pretty midi format
    pmidi_data = mtrack_data.to_pretty_midi(constant_tempo=None)

    # print instruments
    print ('[info] Show {} Insts...'.format(len(pmidi_data.instruments)))
    for x in pmidi_data.instruments:
        print ('[info] MIDI ' + str(x))
    print('')

    # make all notes in Drums2 velocity=99
    for instrument in pmidi_data.instruments:
        #if instrument.program==5:
        if instrument.is_drum:
            for note in instrument.notes:
                note.velocity = 120
        else:
            for note in instrument.notes:
                note.velocity = 50            


    song_name_tmp = all_tracks_mid_flist[x_idx].split('/')[-1][:-15] + '_merged'
                
    # set midi file name to write
    midi_file_name = './output_midi/{}.mid'.format(song_name_tmp)

    # create folder if not exist
    ensure_dir(midi_file_name)

    # write midi file
    pmidi_data.write(midi_file_name)
    print ('[info] \"{}\" is saved.\n\n'.format(midi_file_name))
    
print ('[info] All {} files are saved.'.format(len(song_index_in_list)))

[info] Start processing song: 1 ...
[info] Song index: 00000
[info] Song bars: 256
[info] start\end bar index:  0\256


[dbg] versions=1 first_version_shape=(24576, 128)
[info] saved 00000__drum_only.mid (drum-only)
[info] Start processing song: 2 ...
[info] Song index: 00001
[info] Song bars: 256
[info] start\end bar index:  256\512


[dbg] versions=1 first_version_shape=(24576, 128)
[info] saved 00001__drum_only.mid (drum-only)
[info] Start processing song: 3 ...
[info] Song index: 00002
[info] Song bars: 256
[info] start\end bar index:  512\768


[dbg] versions=1 first_version_shape=(24576, 128)
[info] saved 00002__drum_only.mid (drum-only)
[info] Start processing song: 4 ...
[info] Song index: 00003
[info] Song bars: 256
[info] start\end bar index:  768\1024


[dbg] versions=1 first_version_shape=(24576, 128)
[info] saved 00003__drum_only.mid (drum-only)
[info] Start processing song: 5 ...
[info] Song index: 00004
[info] Song bars: 256
[info] start\end bar index:  1024\1280


...

[dbg] versions=1 first_version_shape=(24576, 128)
[info] saved 00015__drum_only.mid (drum-only)
[info] Start processing song: 17 ...
[info] Song index: 00016
[info] Song bars: 256
[info] start\end bar index:  4096\4352


[dbg] versions=1 first_version_shape=(24576, 128)
[info] saved 00016__drum_only.mid (drum-only)
[info] Start processing song: 18 ...
[info] Song index: 00017
[info] Song bars: 256
[info] start\end bar index:  4352\4608


[dbg] versions=1 first_version_shape=(24576, 128)
[info] saved 00017__drum_only.mid (drum-only)
[info] Start processing song: 19 ...
[info] Song index: 00018
[info] Song bars: 256
[info] start\end bar index:  4608\4864


[dbg] versions=1 first_version_shape=(24576, 128)
[info] saved 00018__drum_only.mid (drum-only)
[info] Start processing song: 20 ...
[info] Song index: 00019
[info] Song bars: 256
[info] start\end bar index:  4864\5120


[dbg] versions=1 first_version_shape=(24576, 128)
[info] saved 00019__drum_only.mid (drum-only)
[info] All 20 files are saved.


# Congratulation ! Now you can find fusion tracks(Original midi + generated drums) under "./output_midi/"

In [9]:
!ls ./output_midi/

00000__drum_only.mid  00007__drum_only.mid  00014__drum_only.mid
00001__drum_only.mid  00008__drum_only.mid  00015__drum_only.mid
00002__drum_only.mid  00009__drum_only.mid  00016__drum_only.mid
00003__drum_only.mid  00010__drum_only.mid  00017__drum_only.mid
00004__drum_only.mid  00011__drum_only.mid  00018__drum_only.mid
00005__drum_only.mid  00012__drum_only.mid  00019__drum_only.mid
00006__drum_only.mid  00013__drum_only.mid


# Use any DAW you like to open the MIDI file, you can see five generated tracks as following.

In [10]:
Image(url="./track22_bj.png",width=1200,height=800)